# 1. Preparação dos Dados

## Projeto de Parceria | Semantix

### Análise de satisfação no e-commerce

Este notebook corresponde à etapa de preparação e entendimento dos dados utilizados no projeto.

Nesta etapa serão realizadas:

- importação das bases;
- inspeção da estrutura dos dados;
- análise de tipos e valores ausentes;
- identificação de duplicidades;
- integração das diferentes tabelas;
- tratamento dos dados;
- criação das variáveis utilizadas nas análises posteriores;
- construção da base analítica final em nível de pedido.

**Unidade de análise definida:** 1 linha = 1 pedido.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Ambiente preparado.")

Ambiente preparado.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

DATA_PATH = Path('/content/drive/MyDrive/semantix-data-project/data/raw')

print("Pasta dos dados:")
print(DATA_PATH)

print("\nArquivos encontrados:")
for arquivo in sorted(DATA_PATH.glob('*.csv')):
    print("-", arquivo.name)

Pasta dos dados:
/content/drive/MyDrive/semantix-data-project/data/raw

Arquivos encontrados:
- olist_customers_dataset.csv
- olist_geolocation_dataset.csv
- olist_order_items_dataset.csv
- olist_order_payments_dataset.csv
- olist_order_reviews_dataset.csv
- olist_orders_dataset.csv
- olist_products_dataset.csv
- olist_sellers_dataset.csv
- product_category_name_translation.csv


In [ ]:
customers = pd.read_csv(DATA_PATH / 'olist_customers_dataset.csv')
geolocation = pd.read_csv(DATA_PATH / 'olist_geolocation_dataset.csv')
order_items = pd.read_csv(DATA_PATH / 'olist_order_items_dataset.csv')
payments = pd.read_csv(DATA_PATH / 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(DATA_PATH / 'olist_order_reviews_dataset.csv')
orders = pd.read_csv(DATA_PATH / 'olist_orders_dataset.csv')
products = pd.read_csv(DATA_PATH / 'olist_products_dataset.csv')
sellers = pd.read_csv(DATA_PATH / 'olist_sellers_dataset.csv')
category_translation = pd.read_csv(
    DATA_PATH / 'product_category_name_translation.csv'
)

print("Bases carregadas com sucesso.")

Bases carregadas com sucesso.


In [ ]:
bases = {
    'customers': customers,
    'geolocation': geolocation,
    'order_items': order_items,
    'payments': payments,
    'reviews': reviews,
    'orders': orders,
    'products': products,
    'sellers': sellers,
    'category_translation': category_translation
}

resumo_bases = pd.DataFrame({
    'base': bases.keys(),
    'linhas': [df.shape[0] for df in bases.values()],
    'colunas': [df.shape[1] for df in bases.values()]
})

resumo_bases

,base,linhas,colunas
0,customers,99441,5
1,geolocation,1000163,5
2,order_items,112650,7
3,payments,103886,5
4,reviews,99224,7
5,orders,99441,8
6,products,32951,9
7,sellers,3095,4
8,category_translation,71,2


## 2. Entendimento da estrutura dos dados

Antes da integração das bases, será analisada a estrutura de cada conjunto de dados, incluindo suas colunas, tipos de variáveis e possíveis chaves de relacionamento.

Essa etapa é necessária porque as tabelas possuem diferentes granularidades. Algumas apresentam um registro por pedido, enquanto outras podem conter múltiplos registros associados ao mesmo `order_id`.

In [ ]:
for nome, df in bases.items():
    print("=" * 80)
    print(f"BASE: {nome.upper()}")
    print("=" * 80)

    print("\nColunas:")
    print(df.columns.tolist())

    print("\nTipos de dados:")
    print(df.dtypes)

    print("\nPrimeiros registros:")
    display(df.head(3))

    print("\n")

BASE: CUSTOMERS

Colunas:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Tipos de dados:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

Primeiros registros:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP




BASE: GEOLOCATION

Colunas:
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Tipos de dados:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

Primeiros registros:


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP




BASE: ORDER_ITEMS

Colunas:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Tipos de dados:
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

Primeiros registros:


...

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0




BASE: SELLERS

Colunas:
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

Tipos de dados:
seller_id                 object
seller_zip_code_prefix     int64
seller_city               object
seller_state              object
dtype: object

Primeiros registros:


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ




BASE: CATEGORY_TRANSLATION

Colunas:
['product_category_name', 'product_category_name_english']

Tipos de dados:
product_category_name            object
product_category_name_english    object
dtype: object

Primeiros registros:


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


### Descrição das bases

As bases utilizadas possuem funções distintas dentro do projeto:

- **customers:** informações de identificação e localização dos clientes;
- **geolocation:** informações geográficas associadas aos CEPs;
- **order_items:** produtos, vendedores, preços e fretes associados aos pedidos;
- **payments:** informações sobre formas e condições de pagamento;
- **reviews:** avaliações realizadas pelos consumidores;
- **orders:** informações gerais e datas relacionadas ao ciclo de cada pedido;
- **products:** características e categorias dos produtos;
- **sellers:** informações de localização dos vendedores;
- **category_translation:** tradução das categorias de produtos.

In [ ]:
chaves = [
    'order_id',
    'customer_id',
    'customer_unique_id',
    'product_id',
    'seller_id',
    'review_id'
]

for chave in chaves:
    presentes = [
        nome
        for nome, df in bases.items()
        if chave in df.columns
    ]

    print(f"{chave}: {presentes}")

order_id: ['order_items', 'payments', 'reviews', 'orders']
customer_id: ['customers', 'orders']
customer_unique_id: ['customers']
product_id: ['order_items', 'products']
seller_id: ['order_items', 'sellers']
review_id: ['reviews']


In [ ]:
analise_order_id = []

for nome, df in bases.items():
    if 'order_id' in df.columns:
        analise_order_id.append({
            'base': nome,
            'linhas': len(df),
            'pedidos_unicos': df['order_id'].nunique(),
            'registros_duplicados_order_id': df.duplicated('order_id').sum()
        })

analise_order_id = pd.DataFrame(analise_order_id)

analise_order_id

,base,linhas,pedidos_unicos,registros_duplicados_order_id
0,order_items,112650,98666,13984
1,payments,103886,99440,4446
2,reviews,99224,98673,551
3,orders,99441,99441,0


### Análise da granularidade por pedido

A verificação do `order_id` demonstrou que a base `orders` possui 99.441 pedidos e nenhum identificador duplicado, sendo adequada como tabela principal da construção da base analítica.

As bases `order_items`, `payments` e `reviews` apresentam múltiplos registros associados ao mesmo pedido:

- `order_items`: 112.650 registros para 98.666 pedidos únicos;
- `payments`: 103.886 registros para 99.440 pedidos únicos;
- `reviews`: 99.224 registros para 98.673 pedidos únicos.

Por esse motivo, essas tabelas não serão integradas diretamente à base `orders`. Primeiro será analisada a origem das repetições e, quando necessário, os registros serão agregados para preservar a unidade de análise definida: **1 linha = 1 pedido**.

In [ ]:
# Exemplos de pedidos com múltiplos itens

pedidos_multiplos_itens = (
    order_items['order_id']
    .value_counts()
    .loc[lambda x: x > 1]
)

print("Pedidos com mais de um registro em order_items:")
display(pedidos_multiplos_itens.head(10))

Pedidos com mais de um registro em order_items:


,count
order_id,
8272b63d03f5f79c56e9e4120aec44ef,21
1b15974a0141d54e36626dca3fdc731a,20
ab14fdcfbe524636d65ee38360e22ce8,20
9ef13efd6949e4573a18964dd1bbe7f5,15
428a2f660dc84138d969ccd69a0ab6d5,15
9bdc4d4c71aa1de4606060929dee888c,14
73c8ab38f07dc94389065f7eba4f297a,14
37ee401157a3a0b28c9c6d0ed8c3b24b,13
2c2a19b5703863c908512d135aa6accc,12


In [ ]:
# Exemplos de pedidos com múltiplos registros de pagamento

pedidos_multiplos_pagamentos = (
    payments['order_id']
    .value_counts()
    .loc[lambda x: x > 1]
)

print("Pedidos com mais de um registro em payments:")
display(pedidos_multiplos_pagamentos.head(10))

Pedidos com mais de um registro em payments:


,count
order_id,
fa65dad1b0e818e3ccc5cb0e39231352,29
ccf804e764ed5650cd8759557269dc13,26
285c2e15bebd4ac83635ccc563dc71f4,22
895ab968e7bb0d5659d16cd74cd1650c,21
fedcd9f7ccdc8cba3a18defedd1a5547,19
ee9ca989fc93ba09a6eddc250ce01742,19
21577126c19bf11a0b91592e5844ba78,15
4bfcba9e084f46c8e3cb49b0fa6e6159,15
3c58bffb70dcf45f12bdf66a3c215905,14


In [ ]:
# Exemplos de pedidos com múltiplas avaliações

pedidos_multiplas_reviews = (
    reviews['order_id']
    .value_counts()
    .loc[lambda x: x > 1]
)

print("Pedidos com mais de um registro em reviews:")
display(pedidos_multiplas_reviews.head(10))

Pedidos com mais de um registro em reviews:


,count
order_id,
03c939fd7fd3b38f8485a0f95798f1f6,3
8e17072ec97ce29f0e1f111e598b0c85,3
c88b1d1b157a9999ce368f218a407141,3
df56136b8031ecd28e200bb18e6ddb2e,3
843be4a0dcdb9716de7652d53af4acab,2
03eba6d9fef8f5b3e811d4b5a7cca9cd,2
c0db7d31ace61fc360a3eaa34dd3457c,2
ceb533871105f7cda81fafc19e1ee38e,2
77ef3216467887c78ddec18de86a58b9,2


In [ ]:
exemplo_order_items = pedidos_multiplos_itens.index[0]

print("Pedido analisado:", exemplo_order_items)

display(
    order_items[
        order_items['order_id'] == exemplo_order_items
    ]
)

Pedido analisado: 8272b63d03f5f79c56e9e4120aec44ef


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
57297,8272b63d03f5f79c56e9e4120aec44ef,1,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57298,8272b63d03f5f79c56e9e4120aec44ef,2,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57299,8272b63d03f5f79c56e9e4120aec44ef,3,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57300,8272b63d03f5f79c56e9e4120aec44ef,4,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57301,8272b63d03f5f79c56e9e4120aec44ef,5,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57302,8272b63d03f5f79c56e9e4120aec44ef,6,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57303,8272b63d03f5f79c56e9e4120aec44ef,7,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57304,8272b63d03f5f79c56e9e4120aec44ef,8,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57305,8272b63d03f5f79c56e9e4120aec44ef,9,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57306,8272b63d03f5f79c56e9e4120aec44ef,10,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89


In [ ]:
exemplo_payments = pedidos_multiplos_pagamentos.index[0]

print("Pedido analisado:", exemplo_payments)

display(
    payments[
        payments['order_id'] == exemplo_payments
    ]
)

Pedido analisado: fa65dad1b0e818e3ccc5cb0e39231352


,order_id,payment_sequential,payment_type,payment_installments,payment_value
4885,fa65dad1b0e818e3ccc5cb0e39231352,27,voucher,1,66.02
9985,fa65dad1b0e818e3ccc5cb0e39231352,4,voucher,1,29.16
14321,fa65dad1b0e818e3ccc5cb0e39231352,1,voucher,1,3.71
17274,fa65dad1b0e818e3ccc5cb0e39231352,9,voucher,1,1.08
19565,fa65dad1b0e818e3ccc5cb0e39231352,10,voucher,1,12.86
23074,fa65dad1b0e818e3ccc5cb0e39231352,2,voucher,1,8.51
24879,fa65dad1b0e818e3ccc5cb0e39231352,25,voucher,1,3.68
28330,fa65dad1b0e818e3ccc5cb0e39231352,5,voucher,1,0.66
29648,fa65dad1b0e818e3ccc5cb0e39231352,6,voucher,1,5.02
32519,fa65dad1b0e818e3ccc5cb0e39231352,11,voucher,1,4.03


In [ ]:
exemplo_reviews = pedidos_multiplas_reviews.index[0]

print("Pedido analisado:", exemplo_reviews)

display(
    reviews[
        reviews['order_id'] == exemplo_reviews
    ]
)

Pedido analisado: 03c939fd7fd3b38f8485a0f95798f1f6


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
8273,b04ed893318da5b863e878cd3d0511df,03c939fd7fd3b38f8485a0f95798f1f6,3,NaN,Um ponto negativo que achei foi a cobrança de ...,2018-03-20 00:00:00,2018-03-21 02:28:23
51527,f4bb9d6dd4fb6dcc2298f0e7b17b8e1e,03c939fd7fd3b38f8485a0f95798f1f6,4,NaN,NaN,2018-03-29 00:00:00,2018-03-30 00:29:09
69438,405eb2ea45e1dbe2662541ae5b47e2aa,03c939fd7fd3b38f8485a0f95798f1f6,3,NaN,Seria ótimo se tivesem entregue os 3 (três) pe...,2018-03-06 00:00:00,2018-03-06 19:50:32


## 3. Avaliação da qualidade dos dados

Nesta etapa serão analisados valores ausentes e duplicidades, com o objetivo de identificar problemas que possam interferir na integração das bases, na análise exploratória e na construção do modelo preditivo.

In [ ]:
resumo_nulos = []

for nome, df in bases.items():
    for coluna in df.columns:
        qtd_nulos = df[coluna].isna().sum()

        if qtd_nulos > 0:
            resumo_nulos.append({
                'base': nome,
                'coluna': coluna,
                'nulos': qtd_nulos,
                'percentual_nulos': round(
                    qtd_nulos / len(df) * 100, 2
                )
            })

resumo_nulos = pd.DataFrame(resumo_nulos)

resumo_nulos.sort_values(
    ['base', 'percentual_nulos'],
    ascending=[True, False]
)

,base,coluna,nulos,percentual_nulos
4,orders,order_delivered_customer_date,2965,2.98
3,orders,order_delivered_carrier_date,1783,1.79
2,orders,order_approved_at,160,0.16
5,products,product_category_name,610,1.85
6,products,product_name_lenght,610,1.85
7,products,product_description_lenght,610,1.85
8,products,product_photos_qty,610,1.85
9,products,product_weight_g,2,0.01
10,products,product_length_cm,2,0.01
11,products,product_height_cm,2,0.01


### Estratégia de tratamento das tabelas com múltiplos registros

A inspeção dos registros demonstrou que a repetição de `order_id` não representa necessariamente duplicidade de dados.

- Em `order_items`, um mesmo pedido pode possuir múltiplos produtos.
- Em `payments`, um mesmo pedido pode possuir múltiplas transações ou sequências de pagamento.
- Em `reviews`, foram identificados pedidos com mais de uma avaliação registrada.

Para preservar a unidade de análise do projeto, essas informações serão agregadas por `order_id` antes da integração com a tabela principal `orders`.

A agregação será realizada preservando informações relevantes para a análise, sem considerar registros legítimos como duplicidades a serem simplesmente excluídas.

In [ ]:
order_items_agg = (
    order_items
    .groupby('order_id')
    .agg(
        quantidade_itens=('order_item_id', 'count'),
        valor_produtos=('price', 'sum'),
        valor_frete=('freight_value', 'sum'),
        quantidade_produtos_unicos=('product_id', 'nunique'),
        quantidade_vendedores=('seller_id', 'nunique')
    )
    .reset_index()
)

print("Dimensão original:", order_items.shape)
print("Dimensão após agregação:", order_items_agg.shape)

display(order_items_agg.head())

Dimensão original: (112650, 7)
Dimensão após agregação: (98666, 6)


,order_id,quantidade_itens,valor_produtos,valor_frete,quantidade_produtos_unicos,quantidade_vendedores
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,1,1
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,1,1
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,1,1
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,1,1


In [ ]:
payments_agg = (
    payments
    .groupby('order_id')
    .agg(
        valor_pagamento=('payment_value', 'sum'),
        quantidade_pagamentos=('payment_sequential', 'count'),
        max_parcelas=('payment_installments', 'max'),
        quantidade_formas_pagamento=('payment_type', 'nunique')
    )
    .reset_index()
)

print("Dimensão original:", payments.shape)
print("Dimensão após agregação:", payments_agg.shape)

display(payments_agg.head())

Dimensão original: (103886, 5)
Dimensão após agregação: (99440, 5)


,order_id,valor_pagamento,quantidade_pagamentos,max_parcelas,quantidade_formas_pagamento
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,3,1
2,000229ec398224ef6ca0657da4fc703e,216.87,1,5,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,2,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,3,1


In [ ]:
reviews['review_creation_date'] = pd.to_datetime(
    reviews['review_creation_date'],
    errors='coerce'
)

reviews['review_answer_timestamp'] = pd.to_datetime(
    reviews['review_answer_timestamp'],
    errors='coerce'
)

In [ ]:
reviews_ordenadas = reviews.sort_values(
    ['order_id', 'review_answer_timestamp', 'review_creation_date']
)

reviews_agg = (
    reviews_ordenadas
    .drop_duplicates(subset='order_id', keep='last')
    [['order_id', 'review_score']]
    .copy()
)

print("Dimensão original:", reviews.shape)
print("Dimensão após tratamento:", reviews_agg.shape)

print(
    "Pedidos duplicados após tratamento:",
    reviews_agg['order_id'].duplicated().sum()
)

display(reviews_agg.head())

Dimensão original: (99224, 7)
Dimensão após tratamento: (98673, 2)
Pedidos duplicados após tratamento: 0


,order_id,review_score
51963,00010242fe8c5a6d1ba2dd792cb16214,5
27823,00018f77f2f0320c557190d7a144bdd3,4
4218,000229ec398224ef6ca0657da4fc703e,5
38844,00024acbcdf0a6daa1e931b038114c75,4
55676,00042b26cf59d7ce69dfabb4e55b4fd9,5


### Tratamento das avaliações múltiplas

Foram identificados pedidos com mais de uma avaliação associada ao mesmo `order_id`.

Como a nota do consumidor será utilizada posteriormente na definição da variável-alvo do modelo preditivo, optei por não calcular a média das avaliações, pois isso poderia gerar valores que não correspondem a uma avaliação efetivamente registrada.

Para os pedidos com múltiplos registros, foi mantida a avaliação mais recente disponível, considerando os campos temporais da base.

Após o tratamento, cada pedido possui no máximo uma avaliação associada.

In [ ]:
validacao_agregacoes = pd.DataFrame({
    'base': [
        'order_items_agg',
        'payments_agg',
        'reviews_agg'
    ],
    'linhas': [
        len(order_items_agg),
        len(payments_agg),
        len(reviews_agg)
    ],
    'order_id_unicos': [
        order_items_agg['order_id'].nunique(),
        payments_agg['order_id'].nunique(),
        reviews_agg['order_id'].nunique()
    ],
    'order_id_duplicados': [
        order_items_agg['order_id'].duplicated().sum(),
        payments_agg['order_id'].duplicated().sum(),
        reviews_agg['order_id'].duplicated().sum()
    ]
})

validacao_agregacoes

,base,linhas,order_id_unicos,order_id_duplicados
0,order_items_agg,98666,98666,0
1,payments_agg,99440,99440,0
2,reviews_agg,98673,98673,0


In [ ]:
display(
    resumo_nulos
    .sort_values(
        'percentual_nulos',
        ascending=False
    )
    .reset_index(drop=True)
)

,base,coluna,nulos,percentual_nulos
0,reviews,review_comment_title,87656,88.34
1,reviews,review_comment_message,58247,58.70
2,orders,order_delivered_customer_date,2965,2.98
3,products,product_name_lenght,610,1.85
4,products,product_category_name,610,1.85
5,products,product_description_lenght,610,1.85
6,products,product_photos_qty,610,1.85
7,orders,order_delivered_carrier_date,1783,1.79
8,orders,order_approved_at,160,0.16
9,products,product_weight_g,2,0.01


### Análise inicial dos valores ausentes

Os valores ausentes não serão removidos automaticamente, pois sua ausência pode ter significados distintos dependendo da variável.

Os campos `review_comment_title` e `review_comment_message` apresentam elevada proporção de valores ausentes, porém não serão utilizados na definição da variável-alvo ou nas análises principais deste projeto.

Na base `orders`, foram identificados valores ausentes em datas relacionadas ao fluxo de entrega. Antes de qualquer exclusão, será investigada a relação desses registros com o status dos pedidos.

Na base `products`, há pequena quantidade de valores ausentes em informações de categoria e características dos produtos, que serão tratadas posteriormente conforme sua utilização na análise.

In [ ]:
colunas_data_orders = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for coluna in colunas_data_orders:
    orders[coluna] = pd.to_datetime(
        orders[coluna],
        errors='coerce'
    )

print("Tipos das colunas de data:")
print(orders[colunas_data_orders].dtypes)

Tipos das colunas de data:
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [ ]:
status_pedidos = (
    orders['order_status']
    .value_counts()
    .rename_axis('status')
    .reset_index(name='quantidade')
)

status_pedidos['percentual'] = (
    status_pedidos['quantidade']
    / len(orders)
    * 100
).round(2)

status_pedidos

,status,quantidade,percentual
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


In [ ]:
analise_nulos_status = (
    orders
    .groupby('order_status')
    .agg(
        quantidade_pedidos=('order_id', 'size'),
        sem_data_aprovacao=(
            'order_approved_at',
            lambda x: x.isna().sum()
        ),
        sem_data_transportadora=(
            'order_delivered_carrier_date',
            lambda x: x.isna().sum()
        ),
        sem_data_entrega_cliente=(
            'order_delivered_customer_date',
            lambda x: x.isna().sum()
        )
    )
    .reset_index()
)

analise_nulos_status

,order_status,quantidade_pedidos,sem_data_aprovacao,sem_data_transportadora,sem_data_entrega_cliente
0,approved,2,0,2,2
1,canceled,625,141,550,619
2,created,5,5,5,5
3,delivered,96478,14,2,8
4,invoiced,314,0,314,314
5,processing,301,0,301,301
6,shipped,1107,0,0,1107
7,unavailable,609,0,609,609


In [ ]:
pedidos_entregues = orders[
    orders['order_status'] == 'delivered'
].copy()

print("Total de pedidos na base:", len(orders))
print("Pedidos com status delivered:", len(pedidos_entregues))

print("\nNulos entre pedidos entregues:")
display(
    pedidos_entregues[
        colunas_data_orders
    ]
    .isna()
    .sum()
    .to_frame('quantidade_nulos')
)

Total de pedidos na base: 99441
Pedidos com status delivered: 96478

Nulos entre pedidos entregues:


,quantidade_nulos
order_purchase_timestamp,0
order_approved_at,14
order_delivered_carrier_date,2
order_delivered_customer_date,8
order_estimated_delivery_date,0


In [ ]:
products_enriched = products.merge(
    category_translation,
    on='product_category_name',
    how='left'
)

products_enriched['product_category_name_english'] = (
    products_enriched['product_category_name_english']
    .fillna('unknown')
)

display(
    products_enriched[
        [
            'product_id',
            'product_category_name',
            'product_category_name_english'
        ]
    ].head()
)

,product_id,product_category_name,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares


In [ ]:
order_items_enriched = order_items.merge(
    products_enriched[
        [
            'product_id',
            'product_category_name',
            'product_category_name_english'
        ]
    ],
    on='product_id',
    how='left'
)

print("Dimensão:", order_items_enriched.shape)

display(order_items_enriched.head())

Dimensão: (112650, 9)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_category_name_english
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,garden_tools


## 4. Definição da população de análise

A análise dos status demonstrou que 96.478 dos 99.441 pedidos da base (97,02%) possuem status `delivered`.

Como o objetivo do projeto é investigar fatores associados à satisfação do consumidor após a experiência de compra, a base analítica será concentrada nos pedidos efetivamente entregues.

Entre os pedidos classificados como entregues, apenas 8 registros não possuem `order_delivered_customer_date`. Como essa variável é necessária para calcular o tempo real de entrega e identificar atrasos, esses registros serão removidos da base analítica.

Não será realizada imputação artificial dessas datas, pois isso poderia introduzir distorções nas variáveis derivadas de desempenho logístico.

In [ ]:
orders_analise = (
    orders[
        (orders['order_status'] == 'delivered') &
        (orders['order_delivered_customer_date'].notna())
    ]
    .copy()
)

print("Pedidos originais:", len(orders))
print("Pedidos entregues:", (orders['order_status'] == 'delivered').sum())
print("Pedidos elegíveis para análise:", len(orders_analise))
print("Pedidos removidos por ausência da data real de entrega:",
      pedidos_entregues['order_delivered_customer_date'].isna().sum())

Pedidos originais: 99441
Pedidos entregues: 96478
Pedidos elegíveis para análise: 96470
Pedidos removidos por ausência da data real de entrega: 8


In [ ]:
orders_analise['tempo_entrega_dias'] = (
    orders_analise['order_delivered_customer_date'] -
    orders_analise['order_purchase_timestamp']
).dt.total_seconds() / 86400

orders_analise['diferenca_prazo_dias'] = (
    orders_analise['order_delivered_customer_date'] -
    orders_analise['order_estimated_delivery_date']
).dt.total_seconds() / 86400

orders_analise['atrasado'] = (
    orders_analise['order_delivered_customer_date'] >
    orders_analise['order_estimated_delivery_date']
).astype(int)

display(
    orders_analise[
        [
            'order_id',
            'order_purchase_timestamp',
            'order_delivered_customer_date',
            'order_estimated_delivery_date',
            'tempo_entrega_dias',
            'diferenca_prazo_dias',
            'atrasado'
        ]
    ].head()
)

,order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,tempo_entrega_dias,diferenca_prazo_dias,atrasado
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,8.436574,-7.107488,0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,13.782037,-5.355729,0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,9.394213,-17.245498,0
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,13.208750,-12.980069,0
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,2.873877,-9.238171,0


In [ ]:
print("Estatísticas do tempo de entrega:")
display(
    orders_analise[
        ['tempo_entrega_dias', 'diferenca_prazo_dias']
    ].describe()
)

print("\nDistribuição de atraso:")
display(
    orders_analise['atrasado']
    .value_counts()
    .rename_axis('atrasado')
    .reset_index(name='quantidade')
)

Estatísticas do tempo de entrega:


,tempo_entrega_dias,diferenca_prazo_dias
count,96470.000000,96470.000000
mean,12.558217,-11.178126
std,9.546156,10.184354
min,0.533414,-146.016123
25%,6.766204,-16.244065
50%,10.217477,-11.948102
75%,15.720182,-6.389815
max,209.628611,188.975081



Distribuição de atraso:


,atrasado,quantidade
0,0,88644
1,1,7826


In [ ]:
categoria_principal = (
    order_items_enriched
    .sort_values(
        ['order_id', 'price'],
        ascending=[True, False]
    )
    .drop_duplicates(
        subset='order_id',
        keep='first'
    )
    [
        [
            'order_id',
            'product_category_name',
            'product_category_name_english'
        ]
    ]
    .rename(
        columns={
            'product_category_name': 'categoria_principal',
            'product_category_name_english': 'categoria_principal_english'
        }
    )
)

categoria_principal['categoria_principal'] = (
    categoria_principal['categoria_principal']
    .fillna('sem_categoria')
)

categoria_principal['categoria_principal_english'] = (
    categoria_principal['categoria_principal_english']
    .fillna('unknown')
)

display(categoria_principal.head())

,order_id,categoria_principal,categoria_principal_english
0,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,pet_shop,pet_shop
2,000229ec398224ef6ca0657da4fc703e,moveis_decoracao,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,perfumaria,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,ferramentas_jardim,garden_tools


In [ ]:
df_analitico = orders_analise.copy()

df_analitico = df_analitico.merge(
    order_items_agg,
    on='order_id',
    how='left',
    validate='one_to_one'
)

df_analitico = df_analitico.merge(
    payments_agg,
    on='order_id',
    how='left',
    validate='one_to_one'
)

df_analitico = df_analitico.merge(
    reviews_agg,
    on='order_id',
    how='left',
    validate='one_to_one'
)

df_analitico = df_analitico.merge(
    categoria_principal,
    on='order_id',
    how='left',
    validate='one_to_one'
)

print("Dimensão da base analítica:", df_analitico.shape)
print("Pedidos únicos:", df_analitico['order_id'].nunique())
print("Order IDs duplicados:", df_analitico['order_id'].duplicated().sum())

display(df_analitico.head())

Dimensão da base analítica: (96470, 23)
Pedidos únicos: 96470
Order IDs duplicados: 0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,tempo_entrega_dias,diferenca_prazo_dias,atrasado,quantidade_itens,valor_produtos,valor_frete,quantidade_produtos_unicos,quantidade_vendedores,valor_pagamento,quantidade_pagamentos,max_parcelas,quantidade_formas_pagamento,review_score,categoria_principal,categoria_principal_english
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.436574,-7.107488,0,1,29.99,8.72,1,1,38.71,3.0,1.0,2.0,4.0,utilidades_domesticas,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.782037,-5.355729,0,1,118.70,22.76,1,1,141.46,1.0,1.0,1.0,4.0,perfumaria,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.394213,-17.245498,0,1,159.90,19.22,1,1,179.12,1.0,3.0,1.0,5.0,automotivo,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.208750,-12.980069,0,1,45.00,27.20,1,1,72.20,1.0,1.0,1.0,5.0,pet_shop,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.873877,-9.238171,0,1,19.90,8.72,1,1,28.62,1.0,1.0,1.0,5.0,papelaria,stationery


In [ ]:
clientes_analise = customers[
    [
        'customer_id',
        'customer_unique_id',
        'customer_zip_code_prefix',
        'customer_city',
        'customer_state'
    ]
].copy()

df_analitico = df_analitico.merge(
    clientes_analise,
    on='customer_id',
    how='left',
    validate='many_to_one'
)

print("Dimensão após integração dos clientes:", df_analitico.shape)
print("Pedidos:", len(df_analitico))
print("Pedidos únicos:", df_analitico['order_id'].nunique())

display(
    df_analitico[
        [
            'order_id',
            'customer_unique_id',
            'customer_city',
            'customer_state'
        ]
    ].head()
)

Dimensão após integração dos clientes: (96470, 27)
Pedidos: 96470
Pedidos únicos: 96470


,order_id,customer_unique_id,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP


### Definição da variável-alvo

A satisfação do consumidor será analisada a partir do campo `review_score`.

Para a construção da variável-alvo do projeto, serão consideradas avaliações de 1 ou 2 estrelas como indicativas de insatisfação. Avaliações de 3 a 5 estrelas serão classificadas como não insatisfeitas.

Essa transformação permite formular o problema como uma tarefa de classificação binária e investigar quais características da experiência de compra estão mais associadas às avaliações negativas.

In [ ]:
df_analitico['insatisfeito'] = np.where(
    df_analitico['review_score'].isna(),
    np.nan,
    (df_analitico['review_score'] <= 2).astype(int)
)

distribuicao_target = (
    df_analitico['insatisfeito']
    .value_counts(dropna=False)
    .rename_axis('insatisfeito')
    .reset_index(name='quantidade')
)

distribuicao_target['percentual'] = (
    distribuicao_target['quantidade']
    / len(df_analitico)
    * 100
).round(2)

distribuicao_target

,insatisfeito,quantidade,percentual
0,0.0,83552,86.61
1,1.0,12272,12.72
2,NaN,646,0.67


## 5. Validação da base analítica

Após a integração das diferentes fontes, será realizada uma validação final da base analítica.

Nesta etapa serão verificados:

- pedidos sem avaliação;
- valores ausentes após as integrações;
- consistência da unidade de análise;
- criação das últimas variáveis derivadas;
- preparação da base que será utilizada na análise exploratória e na modelagem.

In [ ]:
pedidos_sem_avaliacao = df_analitico[
    df_analitico['review_score'].isna()
].copy()

print("Total da base analítica:", len(df_analitico))
print("Pedidos sem avaliação:", len(pedidos_sem_avaliacao))
print(
    "Percentual sem avaliação:",
    round(len(pedidos_sem_avaliacao) / len(df_analitico) * 100, 2),
    "%"
)

display(
    pedidos_sem_avaliacao[
        [
            'order_id',
            'order_purchase_timestamp',
            'customer_state',
            'tempo_entrega_dias',
            'atrasado'
        ]
    ].head()
)

Total da base analítica: 96470
Pedidos sem avaliação: 646
Percentual sem avaliação: 0.67 %


,order_id,order_purchase_timestamp,customer_state,tempo_entrega_dias,atrasado
15,403b97836b0c04a622354cf531062e5f,2018-01-02 19:00:43,RJ,17.276574,0
301,4906eeadde5f70b308c20c4a8f20be02,2017-12-08 04:45:26,RJ,32.555231,1
369,b7a4a9ecb1cd3ef6a3e36a48e200e3be,2017-05-19 18:13:54,SC,19.569306,0
377,59b32faedc12322c672e95ec3716d614,2018-06-27 11:10:11,RS,9.227373,0
395,c2215076050fa358934105b15c34cf3b,2017-07-16 10:04:36,SP,10.435891,0


### Pedidos sem avaliação

Foram identificados 646 pedidos entregues sem `review_score`, correspondendo a aproximadamente 0,67% da base analítica.

Esses pedidos serão mantidos na base geral, pois representam transações válidas. Entretanto, não serão utilizados nas análises que dependem da satisfação do consumidor nem no treinamento do modelo preditivo, já que não existe informação suficiente para determinar a variável-alvo `insatisfeito`.

Dessa forma, evita-se classificar artificialmente consumidores sem avaliação como satisfeitos ou insatisfeitos.

In [ ]:
nulos_df_analitico = (
    df_analitico
    .isna()
    .sum()
    .reset_index()
)

nulos_df_analitico.columns = ['coluna', 'nulos']

nulos_df_analitico['percentual'] = (
    nulos_df_analitico['nulos']
    / len(df_analitico)
    * 100
).round(2)

nulos_df_analitico = (
    nulos_df_analitico[
        nulos_df_analitico['nulos'] > 0
    ]
    .sort_values(
        'percentual',
        ascending=False
    )
    .reset_index(drop=True)
)

display(nulos_df_analitico)

,coluna,nulos,percentual
0,review_score,646,0.67
1,insatisfeito,646,0.67
2,order_approved_at,14,0.01
3,order_delivered_carrier_date,1,0.00
4,quantidade_pagamentos,1,0.00
5,valor_pagamento,1,0.00
6,quantidade_formas_pagamento,1,0.00
7,max_parcelas,1,0.00


In [ ]:
# Percentual do frete em relação ao valor dos produtos

df_analitico['frete_percentual'] = np.where(
    df_analitico['valor_produtos'] > 0,
    (
        df_analitico['valor_frete']
        / df_analitico['valor_produtos']
        * 100
    ),
    np.nan
)

# Valor total do pedido considerando produtos + frete

df_analitico['valor_total_pedido'] = (
    df_analitico['valor_produtos']
    + df_analitico['valor_frete']
)

# Componentes temporais da compra

df_analitico['ano_compra'] = (
    df_analitico['order_purchase_timestamp'].dt.year
)

df_analitico['mes_compra'] = (
    df_analitico['order_purchase_timestamp'].dt.month
)

df_analitico['dia_semana_compra'] = (
    df_analitico['order_purchase_timestamp']
    .dt.day_name()
)

df_analitico['hora_compra'] = (
    df_analitico['order_purchase_timestamp'].dt.hour
)

print("Novas variáveis criadas com sucesso.")

display(
    df_analitico[
        [
            'order_id',
            'valor_produtos',
            'valor_frete',
            'frete_percentual',
            'valor_total_pedido',
            'ano_compra',
            'mes_compra',
            'dia_semana_compra',
            'hora_compra'
        ]
    ].head()
)

Novas variáveis criadas com sucesso.


,order_id,valor_produtos,valor_frete,frete_percentual,valor_total_pedido,ano_compra,mes_compra,dia_semana_compra,hora_compra
0,e481f51cbdc54678b7cc49136f2d6af7,29.99,8.72,29.076359,38.71,2017,10,Monday,10
1,53cdb2fc8bc7dce0b6741e2150273451,118.70,22.76,19.174389,141.46,2018,7,Tuesday,20
2,47770eb9100c2d0c44946d9cf07ec65d,159.90,19.22,12.020013,179.12,2018,8,Wednesday,8
3,949d5b44dbf5de918fe9c16f97b45f8a,45.00,27.20,60.444444,72.20,2017,11,Saturday,19
4,ad21c59c0840e6cb83a9ceb5573f8159,19.90,8.72,43.819095,28.62,2018,2,Tuesday,21


In [ ]:
df_modelagem = (
    df_analitico[
        df_analitico['review_score'].notna()
    ]
    .copy()
)

df_modelagem['insatisfeito'] = (
    df_modelagem['insatisfeito']
    .astype(int)
)

print("Base analítica completa:", len(df_analitico))
print("Base com avaliação conhecida:", len(df_modelagem))
print(
    "Pedidos removidos apenas da modelagem:",
    len(df_analitico) - len(df_modelagem)
)

print("\nDistribuição da variável-alvo:")
display(
    df_modelagem['insatisfeito']
    .value_counts()
    .rename_axis('insatisfeito')
    .reset_index(name='quantidade')
)

Base analítica completa: 96470
Base com avaliação conhecida: 95824
Pedidos removidos apenas da modelagem: 646

Distribuição da variável-alvo:


,insatisfeito,quantidade
0,0,83552
1,1,12272


In [ ]:
validacao_final = pd.DataFrame({
    'verificacao': [
        'Linhas da base analítica',
        'Pedidos únicos',
        'Order IDs duplicados',
        'Pedidos com avaliação',
        'Pedidos sem avaliação'
    ],
    'resultado': [
        len(df_analitico),
        df_analitico['order_id'].nunique(),
        df_analitico['order_id'].duplicated().sum(),
        df_analitico['review_score'].notna().sum(),
        df_analitico['review_score'].isna().sum()
    ]
})

validacao_final

,verificacao,resultado
0,Linhas da base analítica,96470
1,Pedidos únicos,96470
2,Order IDs duplicados,0
3,Pedidos com avaliação,95824
4,Pedidos sem avaliação,646


In [ ]:
PROCESSED_PATH = DATA_PATH.parent / 'processed'

PROCESSED_PATH.mkdir(
    parents=True,
    exist_ok=True
)

print("Pasta criada:")
print(PROCESSED_PATH)

Pasta criada:
/content/drive/MyDrive/semantix-data-project/data/processed


In [ ]:
arquivo_analitico = (
    PROCESSED_PATH
    / 'olist_analytical_dataset.csv'
)

arquivo_modelagem = (
    PROCESSED_PATH
    / 'olist_modeling_dataset.csv'
)

df_analitico.to_csv(
    arquivo_analitico,
    index=False
)

df_modelagem.to_csv(
    arquivo_modelagem,
    index=False
)

print("Arquivos salvos com sucesso:")
print("-", arquivo_analitico.name)
print("-", arquivo_modelagem.name)

Arquivos salvos com sucesso:
- olist_analytical_dataset.csv
- olist_modeling_dataset.csv


In [ ]:
print(
    "Base analítica existe:",
    arquivo_analitico.exists()
)

print(
    "Base de modelagem existe:",
    arquivo_modelagem.exists()
)

print("\nTamanhos dos arquivos:")

print(
    arquivo_analitico.name,
    round(
        arquivo_analitico.stat().st_size
        / 1024**2,
        2
    ),
    "MB"
)

print(
    arquivo_modelagem.name,
    round(
        arquivo_modelagem.stat().st_size
        / 1024**2,
        2
    ),
    "MB"
)

Base analítica existe: True
Base de modelagem existe: True

Tamanhos dos arquivos:
olist_analytical_dataset.csv 34.77 MB
olist_modeling_dataset.csv 34.36 MB


## 6. Conclusão da preparação dos dados

A etapa de preparação resultou na construção de uma base analítica em nível de pedido, preservando a granularidade definida para o projeto.

Foram realizadas:

- importação e inspeção das nove bases públicas;
- identificação das chaves de relacionamento;
- análise de valores ausentes e registros múltiplos;
- agregação das tabelas de itens e pagamentos;
- tratamento de pedidos com múltiplas avaliações;
- seleção dos pedidos efetivamente entregues;
- integração das informações de pedidos, clientes, produtos, pagamentos e avaliações;
- criação de variáveis relacionadas ao desempenho logístico, valores do pedido e comportamento temporal;
- definição da variável-alvo `insatisfeito`;
- criação de uma base específica para as etapas de análise exploratória e modelagem.

A base preparada será utilizada na próxima etapa do projeto: **Análise Exploratória de Dados (EDA)**.